In [14]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import confusion_matrix

In [15]:
class SingleLayer(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, 8),
            nn.ReLU(),
            #nn.Dropout(0.25), # works slightly better without the dropout layer, and considering we're already doing mini-batching and a train-test split, I'm not too concerned about overfitting
            nn.Linear(8, 2))
    def forward(self, x):
        return(self.sequential(x))

In [16]:
mystery_data = pd.read_csv("../../data/FP_Data.csv")

mystery_data_onehot = pd.get_dummies(mystery_data)

y = mystery_data_onehot.pop("y")
y = np.array([1 if obs >= 70 else 0 for obs in y])
X = mystery_data_onehot

X = normalize(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=28)

In [17]:
model = SingleLayer(11)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1) # This is, as far as I seen, the most widely used optimizer, though it is not what is used in the textbook
loss_fn = nn.CrossEntropyLoss()
#log_softmax = nn.functional.log_softmax(dim = 1, dtype = torch.float32)

epochs = 50
batch_size = 32

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

for epoch in range(epochs):
    model.train()

    permutation = torch.randperm(X_train.size(0))
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        X_batch, y_batch = X_train[indices], y_train[indices]

        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test)
        val_loss = loss_fn(val_output, y_test)
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {val_loss.item():.4f}")

Epoch 1/50 | Val Loss: 0.3282
Epoch 2/50 | Val Loss: 0.2556
Epoch 3/50 | Val Loss: 0.2717
Epoch 4/50 | Val Loss: 0.3259
Epoch 5/50 | Val Loss: 0.3534
Epoch 6/50 | Val Loss: 0.3182
Epoch 7/50 | Val Loss: 0.3443
Epoch 8/50 | Val Loss: 0.3579
Epoch 9/50 | Val Loss: 0.3495
Epoch 10/50 | Val Loss: 0.4301
Epoch 11/50 | Val Loss: 0.4643
Epoch 12/50 | Val Loss: 0.4092
Epoch 13/50 | Val Loss: 0.3307
Epoch 14/50 | Val Loss: 0.3116
Epoch 15/50 | Val Loss: 0.3761
Epoch 16/50 | Val Loss: 0.3755
Epoch 17/50 | Val Loss: 0.3335
Epoch 18/50 | Val Loss: 0.3656
Epoch 19/50 | Val Loss: 0.3640
Epoch 20/50 | Val Loss: 0.4052
Epoch 21/50 | Val Loss: 0.4033
Epoch 22/50 | Val Loss: 0.4478
Epoch 23/50 | Val Loss: 0.4629
Epoch 24/50 | Val Loss: 0.4566
Epoch 25/50 | Val Loss: 0.5594
Epoch 26/50 | Val Loss: 0.4857
Epoch 27/50 | Val Loss: 0.5660
Epoch 28/50 | Val Loss: 0.5887
Epoch 29/50 | Val Loss: 0.5417
Epoch 30/50 | Val Loss: 0.5576
Epoch 31/50 | Val Loss: 0.5980
Epoch 32/50 | Val Loss: 0.6145
Epoch 33/50 | Val

In [18]:
model.eval()
y_test = y_test.detach().numpy()
with torch.no_grad():
    test_pred = model(X_test).argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()
    confusion = confusion_matrix(y_test, test_pred)
    print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.8400


## 10-Fold Cross Validation

In [19]:
df = pd.read_csv("../04-cv/10-fold-cv.csv")

torch.manual_seed(28)
np.random.seed(28)

df_onehot = pd.get_dummies(df.drop(columns=["fold"]))
y = df_onehot.pop("y")
y = np.array([1 if obs >= 70 else 0 for obs in y])
X = normalize(df_onehot)

k = 10
epochs = 50
batch_size = 32
nn_cv_results = []

In [20]:
for fold in range(1, k + 1):

    train_mask = (df["fold"] != fold).values
    test_mask = (df["fold"] == fold).values

    X_train = torch.tensor(X[train_mask], dtype=torch.float32)
    X_test = torch.tensor(X[test_mask], dtype=torch.float32)
    y_train = torch.tensor(np.asarray(y[train_mask]), dtype=torch.long)
    y_test = torch.tensor(np.asarray(y[test_mask]), dtype=torch.long)

    model = SingleLayer(11)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        permutation = torch.randperm(X_train.size(0))
        for i in range(0, X_train.size(0), batch_size):
            indices = permutation[i:i+batch_size]
            X_batch, y_batch = X_train[indices], y_train[indices]

            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_output = model(X_test)
            val_loss = loss_fn(val_output, y_test)

    model.eval()
    y_test_np = y_test.detach().numpy()
    with torch.no_grad():
        test_pred = model(X_test).argmax(dim=1)
        accuracy = (test_pred == y_test).float().mean().item()
        confusion = confusion_matrix(y_test_np, test_pred)

    tn, fp, fn, tp = confusion.ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    nn_cv_results.append({"fold": fold, "model": "NeuralNetwork", "accuracy": accuracy, "fpr": fpr, "fnr": fnr})

nn_cv_df = pd.DataFrame(nn_cv_results)
nn_cv_df.to_csv("../04-cv/class_nn_cv_results.csv", index=False)
print(nn_cv_df)

   fold          model  accuracy       fpr       fnr
0     1  NeuralNetwork  0.761905  0.235294  0.250000
1     2  NeuralNetwork  0.809524  0.111111  0.666667
2     3  NeuralNetwork  0.761905  0.058824  1.000000
3     4  NeuralNetwork  0.857143  0.125000  0.200000
4     5  NeuralNetwork  0.714286  0.250000  0.400000
5     6  NeuralNetwork  0.750000  0.210526  1.000000
6     7  NeuralNetwork  0.750000  0.153846  0.428571
7     8  NeuralNetwork  0.842105  0.200000  0.000000
8     9  NeuralNetwork  0.888889  0.062500  0.500000
9    10  NeuralNetwork  0.777778  0.125000  1.000000
